# Phase 6 — Experimentation and A/B Testing

This phase extends the retention analysis from prediction into causal experimentation.

The phase has two distinct components:

1. **Retention-offer experiment design**
   - design a hypothetical randomized retention campaign for the online retailer;
   - define the treatment and control groups;
   - select a primary metric and guardrail metrics;
   - perform power analysis and estimate the required sample size.

2. **Public A/B experiment analysis**
   - analyze the public Cookie Cats randomized experiment;
   - verify experiment integrity with a sample-ratio check;
   - estimate treatment effects;
   - calculate confidence intervals and effect sizes;
   - make a ship / don't-ship recommendation.

The retailer experiment is a proposed experimental design only and does not represent a real campaign.

Causal conclusions are restricted to the public randomized experiment.

## 6.1 Public Experiment Dataset Inspection

The public Cookie Cats dataset is inspected before any hypothesis testing.

The purpose of this step is to verify:

- dataset size and structure;
- treatment and control labels;
- missing values and duplicate observations;
- retention outcome fields;
- whether each player appears only once;
- basic balance between experimental groups.

No treatment-effect conclusions are drawn at this stage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats


PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cookie_cats.csv"
)


cookie_cats = pd.read_csv(
    DATA_PATH
)


print(
    "Dataset shape:",
    cookie_cats.shape,
)

print(
    "\nColumns:"
)

print(
    cookie_cats.columns.tolist()
)

print(
    "\nData types:"
)

display(
    cookie_cats.dtypes.to_frame(
        name="dtype"
    )
)

print(
    "\nMissing values:"
)

display(
    cookie_cats.isna().sum().to_frame(
        name="missing"
    )
)

print(
    "\nDuplicate rows:",
    cookie_cats.duplicated().sum(),
)

print(
    "Duplicate user IDs:",
    cookie_cats["userid"].duplicated().sum(),
)

print(
    "\nExperiment group counts:"
)

display(
    cookie_cats["version"]
    .value_counts()
    .to_frame(
        name="players"
    )
)

print(
    "\nRetention outcome values:"
)

for column in [
    "retention_1",
    "retention_7",
]:
    print(
        f"\n{column}:"
    )

    print(
        cookie_cats[column]
        .value_counts(
            dropna=False
        )
    )


display(
    cookie_cats.head()
)

Dataset shape: (90189, 5)

Columns:
['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']

Data types:


,dtype
userid,int64
version,str
sum_gamerounds,int64
retention_1,bool
retention_7,bool



Missing values:


,missing
userid,0
version,0
sum_gamerounds,0
retention_1,0
retention_7,0



Duplicate rows: 0
Duplicate user IDs: 0

Experiment group counts:


,players
version,
gate_40,45489
gate_30,44700



Retention outcome values:

retention_1:
retention_1
False    50036
True     40153
Name: count, dtype: int64

retention_7:
retention_7
False    73408
True     16781
Name: count, dtype: int64


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


### Dataset Inspection — Interpretation

The public experiment dataset contains **90,189 unique players** and five variables.

No missing values, duplicate rows, or duplicate user IDs were identified.

The experiment contains two variants:

- **gate_30:** 44,700 players
- **gate_40:** 45,489 players

The dataset provides two binary retention outcomes:

- **1-day retention**
- **7-day retention**

For the primary experiment analysis:

- **gate_30** is treated as the control group;
- **gate_40** is treated as the treatment group;
- **7-day retention** is used as the primary outcome because it represents a stronger medium-term engagement measure than next-day return.

One-day retention is retained as a secondary outcome.

Before estimating treatment effects, experiment allocation is checked for a potential sample-ratio mismatch.

## 6.2 Sample-Ratio Check

Before analyzing retention outcomes, the experiment allocation is checked for a potential **sample-ratio mismatch (SRM)**.

Under an equal-allocation randomized experiment, approximately 50% of players would be expected in each variant.

A chi-square goodness-of-fit test compares the observed group counts with the expected 50/50 split.

Hypotheses:

- **H₀:** the observed allocation is consistent with a 50/50 split;
- **H₁:** the observed allocation differs from a 50/50 split.

A statistically significant result would not by itself prove that randomization failed, but it would flag a potential experiment-integrity issue that should be considered when interpreting the treatment-effect analysis.

In [2]:
group_counts = (
    cookie_cats["version"]
    .value_counts()
    .reindex(
        [
            "gate_30",
            "gate_40",
        ]
    )
)


total_players = (
    group_counts.sum()
)

expected_count = (
    total_players / 2
)

expected_counts = np.array(
    [
        expected_count,
        expected_count,
    ]
)


chi2_stat, srm_p_value = (
    stats.chisquare(
        f_obs=group_counts.values,
        f_exp=expected_counts,
    )
)


allocation_table = pd.DataFrame(
    {
        "group": group_counts.index,
        "observed": group_counts.values,
        "expected_50_50": expected_counts,
        "observed_share": (
            group_counts.values
            / total_players
        ),
    }
)


allocation_table[
    "observed_share"
] = (
    allocation_table[
        "observed_share"
    ]
    * 100
)


display(
    allocation_table.round(4)
)


print(
    "Chi-square statistic:",
    f"{chi2_stat:.4f}",
)

print(
    "SRM p-value:",
    f"{srm_p_value:.6f}",
)

,group,observed,expected_50_50,observed_share
0,gate_30,44700,45094.5,49.5626
1,gate_40,45489,45094.5,50.4374


Chi-square statistic: 6.9024
SRM p-value: 0.008608


### Sample-Ratio Check — Interpretation

The observed experiment allocation is:

- **gate_30:** 44,700 players (49.56%)
- **gate_40:** 45,489 players (50.44%)

Under an assumed 50/50 allocation, the chi-square goodness-of-fit test produced:

- **Chi-square statistic:** 6.9024
- **p-value:** 0.0086

The numerical imbalance is small: the two allocation shares differ by approximately **0.87 percentage points**.

For the sample-ratio-mismatch diagnostic, a stricter significance threshold of **0.001** is used to avoid treating small random allocation differences as experiment-integrity failures.

Because **p = 0.0086 > 0.001**, the allocation does not trigger an SRM flag under this threshold.

The check still assumes that the experiment was intended to use approximately equal allocation, so this assumption should be documented when interpreting the experiment.

## 6.3 Retention Rates and Observed Treatment Effects

Before conducting hypothesis tests, retention outcomes are summarized separately for the control and treatment groups.

The primary outcome is **7-day retention**, while **1-day retention** is treated as a secondary outcome.

For each metric, the analysis reports:

- retention rate in the control group;
- retention rate in the treatment group;
- absolute percentage-point difference;
- relative change compared with the control group.

Statistical significance is evaluated only after the observed effect sizes have been inspected.

In [3]:
retention_summary = (
    cookie_cats
    .groupby("version")
    .agg(
        players=("userid", "size"),
        retention_1_rate=("retention_1", "mean"),
        retention_7_rate=("retention_7", "mean"),
    )
    .reindex(
        [
            "gate_30",
            "gate_40",
        ]
    )
)


control_1 = retention_summary.loc[
    "gate_30",
    "retention_1_rate",
]

treatment_1 = retention_summary.loc[
    "gate_40",
    "retention_1_rate",
]

control_7 = retention_summary.loc[
    "gate_30",
    "retention_7_rate",
]

treatment_7 = retention_summary.loc[
    "gate_40",
    "retention_7_rate",
]


effect_summary = pd.DataFrame(
    {
        "metric": [
            "1-day retention",
            "7-day retention",
        ],
        "control_rate": [
            control_1,
            control_7,
        ],
        "treatment_rate": [
            treatment_1,
            treatment_7,
        ],
    }
)


effect_summary[
    "absolute_difference_pp"
] = (
    (
        effect_summary["treatment_rate"]
        - effect_summary["control_rate"]
    )
    * 100
)


effect_summary[
    "relative_change_pct"
] = (
    (
        effect_summary["treatment_rate"]
        / effect_summary["control_rate"]
        - 1
    )
    * 100
)


display(
    retention_summary.round(4)
)

display(
    effect_summary.round(4)
)

,players,retention_1_rate,retention_7_rate
version,,,
gate_30,44700,0.4482,0.1902
gate_40,45489,0.4423,0.1820


,metric,control_rate,treatment_rate,absolute_difference_pp,relative_change_pct
0,1-day retention,0.4482,0.4423,-0.5905,-1.3176
1,7-day retention,0.1902,0.1820,-0.8201,-4.3119


### Retention Rates — Interpretation

Observed retention is lower in the treatment group for both measured outcomes.

For **1-day retention**:

- Control (`gate_30`): **44.82%**
- Treatment (`gate_40`): **44.23%**
- Absolute difference: **-0.59 percentage points**
- Relative change: **-1.32%**

For the primary **7-day retention** outcome:

- Control (`gate_30`): **19.02%**
- Treatment (`gate_40`): **18.20%**
- Absolute difference: **-0.82 percentage points**
- Relative change: **-4.31%**

The observed direction therefore favors the control condition.

However, these descriptive differences alone do not establish that the treatment caused a meaningful change. Statistical inference is required to quantify uncertainty around the treatment effect.

## 6.4 Primary Treatment-Effect Test

The primary outcome is **7-day retention**.

A two-sample proportion z-test is used because:

- the outcome is binary;
- the treatment and control groups are independent;
- both groups contain large samples;
- the expected numbers of retained and non-retained users are sufficiently large for the normal approximation.

Hypotheses:

- **H₀:** 7-day retention is equal between `gate_40` and `gate_30`;
- **H₁:** 7-day retention differs between `gate_40` and `gate_30`.

The analysis reports:

- the two-sided p-value;
- a 95% confidence interval for the treatment-minus-control difference;
- Cohen's h as a standardized effect-size measure.

The absolute percentage-point difference remains the primary business-facing effect measure.

In [4]:
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)


control = cookie_cats[
    cookie_cats["version"] == "gate_30"
]

treatment = cookie_cats[
    cookie_cats["version"] == "gate_40"
]


control_successes = int(
    control["retention_7"].sum()
)

treatment_successes = int(
    treatment["retention_7"].sum()
)


control_n = len(control)
treatment_n = len(treatment)


# Treatment is placed first so the tested
# difference is treatment - control.
z_stat, p_value = proportions_ztest(
    count=[
        treatment_successes,
        control_successes,
    ],
    nobs=[
        treatment_n,
        control_n,
    ],
    alternative="two-sided",
)


ci_low, ci_high = confint_proportions_2indep(
    count1=treatment_successes,
    nobs1=treatment_n,
    count2=control_successes,
    nobs2=control_n,
    method="wald",
    compare="diff",
    alpha=0.05,
)


cohens_h = proportion_effectsize(
    treatment_7,
    control_7,
)


print(
    "Control retained:",
    control_successes,
    "/",
    control_n,
)

print(
    "Treatment retained:",
    treatment_successes,
    "/",
    treatment_n,
)

print(
    "Z statistic:",
    f"{z_stat:.4f}",
)

print(
    "Two-sided p-value:",
    f"{p_value:.6f}",
)

print(
    "95% CI for treatment - control:",
    f"[{ci_low * 100:.4f}, {ci_high * 100:.4f}] percentage points",
)

print(
    "Cohen's h:",
    f"{cohens_h:.4f}",
)

Control retained: 8502 / 44700
Treatment retained: 8279 / 45489
Z statistic: -3.1644
Two-sided p-value: 0.001554
95% CI for treatment - control: [-1.3282, -0.3121] percentage points
Cohen's h: -0.0211


### Primary Treatment Effect — Interpretation

The `gate_40` treatment produced lower 7-day retention than the `gate_30` control.

Observed 7-day retention was:

- **Control (`gate_30`):** 19.02%
- **Treatment (`gate_40`):** 18.20%
- **Absolute treatment effect:** approximately **-0.82 percentage points**
- **Relative change:** approximately **-4.31%**

The two-sample proportion z-test produced:

- **Z statistic:** -3.1644
- **Two-sided p-value:** 0.001554
- **95% confidence interval for treatment - control:** -1.33 to -0.31 percentage points
- **Cohen's h:** -0.0211

The confidence interval lies entirely below zero, providing evidence that moving the gate from level 30 to level 40 reduced 7-day retention in this experiment.

The standardized effect size is small, showing that the difference is statistically detectable partly because of the large sample size. The business relevance should therefore be interpreted using the absolute and relative retention changes rather than the p-value alone.

Under the observed experiment results, the evidence does **not** support shipping `gate_40` as an improvement to 7-day retention.

This causal interpretation is made cautiously because the earlier sample-ratio check produced a potential allocation-integrity flag under an assumed 50/50 assignment.

## 6.5 Secondary Outcome — 1-Day Retention

One-day retention is evaluated as a secondary outcome.

This analysis is secondary to the prespecified 7-day retention test and is used to understand whether the observed treatment effect appears immediately after exposure or becomes more pronounced over time.

In [5]:
control_successes_1 = int(
    control["retention_1"].sum()
)

treatment_successes_1 = int(
    treatment["retention_1"].sum()
)


z_stat_1, p_value_1 = proportions_ztest(
    count=[
        treatment_successes_1,
        control_successes_1,
    ],
    nobs=[
        treatment_n,
        control_n,
    ],
    alternative="two-sided",
)


ci_low_1, ci_high_1 = confint_proportions_2indep(
    count1=treatment_successes_1,
    nobs1=treatment_n,
    count2=control_successes_1,
    nobs2=control_n,
    method="wald",
    compare="diff",
    alpha=0.05,
)


cohens_h_1 = proportion_effectsize(
    treatment_1,
    control_1,
)


print(
    "Control retained:",
    control_successes_1,
    "/",
    control_n,
)

print(
    "Treatment retained:",
    treatment_successes_1,
    "/",
    treatment_n,
)

print(
    "Z statistic:",
    f"{z_stat_1:.4f}",
)

print(
    "Two-sided p-value:",
    f"{p_value_1:.6f}",
)

print(
    "95% CI for treatment - control:",
    f"[{ci_low_1 * 100:.4f}, {ci_high_1 * 100:.4f}] percentage points",
)

print(
    "Cohen's h:",
    f"{cohens_h_1:.4f}",
)

Control retained: 20034 / 44700
Treatment retained: 20119 / 45489
Z statistic: -1.7841
Two-sided p-value: 0.074410
95% CI for treatment - control: [-1.2392, 0.0582] percentage points
Cohen's h: -0.0119


### Secondary Outcome — Interpretation

One-day retention is slightly lower in the treatment group:

- **Control (`gate_30`):** 44.82%
- **Treatment (`gate_40`):** 44.23%
- **Absolute difference:** approximately **-0.59 percentage points**
- **Relative change:** approximately **-1.32%**

The two-sample proportion z-test produced:

- **Z statistic:** -1.7841
- **Two-sided p-value:** 0.0744
- **95% confidence interval:** -1.24 to +0.06 percentage points
- **Cohen's h:** -0.0119

The confidence interval includes zero and the p-value is above 0.05.

The experiment therefore does not provide sufficient evidence that `gate_40` changed 1-day retention.

The observed direction is negative, but the effect is small and statistically uncertain.

This contrasts with the primary 7-day outcome, where the treatment produced a statistically detectable reduction in retention.

## 6.6 Public Experiment Recommendation

The primary decision metric is 7-day retention.

The experiment shows:

- `gate_30` 7-day retention: **19.02%**
- `gate_40` 7-day retention: **18.20%**
- absolute treatment effect: approximately **-0.82 percentage points**
- relative change: approximately **-4.31%**
- 95% confidence interval: **-1.33 to -0.31 percentage points**
- two-sided p-value: **0.0016**

The treatment therefore produced a statistically detectable reduction in the primary retention metric.

One-day retention also moved in the negative direction, but that secondary effect was not statistically distinguishable from zero.

### Decision

**Do not ship `gate_40` if the objective is to improve player retention.**

The primary randomized outcome indicates that moving the gate from level 30 to level 40 reduces 7-day retention.

The standardized effect size is small, but the absolute reduction is relevant when applied across a large user population.

Because this is a randomized public A/B experiment, the treatment comparison can support a causal interpretation within the experiment's population and design.

This conclusion applies only to the Cookie Cats experiment and should not be interpreted as evidence about the retailer's customers or about retention offers in the main project dataset.

## 6.7 Retailer Retention-Offer Experiment Design

The main retail dataset does not contain a real retention campaign or randomized intervention.

This section therefore designs a **hypothetical randomized controlled experiment** that the retailer could run before rolling out a retention offer.

### Target Population

Customers identified as eligible for retention outreach would be randomly assigned after the campaign cutoff.

The experiment would include only customers who satisfy the campaign's eligibility rules before randomization.

### Experimental Groups

- **Control:** no retention offer; customers continue with the normal experience.
- **Treatment:** receive a retention offer intended to encourage another purchase.

Assignment should be randomized at the **customer level**, with each customer appearing in only one experimental group.

A **50/50 allocation** is used to maximize statistical power for a fixed total sample size.

### Primary Metric

The primary outcome is:

**Repeat purchase within 30 days of randomization**

For each customer:

- `1` = at least one qualifying merchandise purchase within the following 30 days;
- `0` = no qualifying merchandise purchase within the following 30 days.

The treatment effect would be measured as the difference in 30-day repeat-purchase rates between treatment and control.

### Guardrail Metrics

The experiment should also monitor metrics that could reveal undesirable side effects:

- average merchandise revenue per randomized customer;
- average order value among purchasers;
- cancellation / return rate, if operational data is available;
- number of orders per randomized customer;
- offer cost per treated customer;
- net incremental value after campaign cost.

The primary success criterion is an improvement in 30-day repeat purchase without material deterioration in the guardrails.

### Statistical Design

The primary treatment comparison would use a two-sample proportion test with:

- **two-sided significance level:** 5%;
- **statistical power:** 80%;
- **50/50 treatment allocation**.

The required sample size depends on:

1. the expected control-group repeat-purchase rate;
2. the minimum treatment effect considered commercially meaningful.

Because this project contains no real campaign, the experiment design and power calculation are planning exercises rather than evidence of actual business impact.

## 6.8 Historical 30-Day Repeat-Purchase Baseline

The hypothetical retailer experiment requires an expected control-group repeat-purchase rate for power analysis.

Rather than selecting an arbitrary baseline, the historical retail data is used to estimate the proportion of eligible customers who make another qualifying merchandise purchase within 30 days after a historical cutoff.

The estimate is used only for experiment planning.

It is not treated as a causal control-group outcome because the historical retail data was not generated by a randomized experiment.

In [8]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine


load_dotenv(
    PROJECT_ROOT / ".env"
)


DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:"
    f"{os.getenv('DB_PORT')}/"
    f"{os.getenv('DB_NAME')}"
)


engine = create_engine(
    DATABASE_URL
)


print(
    "PostgreSQL connection configured."
)

PostgreSQL connection configured.


In [10]:
from sqlalchemy import text


baseline_cutoff = pd.Timestamp(
    "2011-06-01"
)


baseline_query = text(
    """
    WITH eligible_customers AS (
        SELECT DISTINCT
            customer_id
        FROM retail_transactions
        WHERE invoice_date >= :observation_start
          AND invoice_date < :cutoff
    ),

    followup_purchases AS (
        SELECT DISTINCT
            customer_id
        FROM retail_transactions
        WHERE invoice_date >= :cutoff
          AND invoice_date < :followup_end
    )

    SELECT
        e.customer_id,
        CASE
            WHEN f.customer_id IS NOT NULL THEN 1
            ELSE 0
        END AS repeat_purchase_30d
    FROM eligible_customers e
    LEFT JOIN followup_purchases f
        ON e.customer_id = f.customer_id
    ORDER BY e.customer_id;
    """
)


baseline_customers = pd.read_sql(
    baseline_query,
    engine,
    params={
        "observation_start": (
            baseline_cutoff
            - pd.Timedelta(days=180)
        ),
        "cutoff": baseline_cutoff,
        "followup_end": (
            baseline_cutoff
            + pd.Timedelta(days=30)
        ),
    },
)


baseline_repeat_rate = (
    baseline_customers[
        "repeat_purchase_30d"
    ].mean()
)


print(
    "Historical cutoff:",
    baseline_cutoff.date(),
)

print(
    "Eligible customers:",
    len(baseline_customers),
)

print(
    "30-day repeat purchasers:",
    baseline_customers[
        "repeat_purchase_30d"
    ].sum(),
)

print(
    "Historical 30-day repeat-purchase rate:",
    f"{baseline_repeat_rate * 100:.2f}%",
)

Historical cutoff: 2011-06-01
Eligible customers: 2658
30-day repeat purchasers: 745
Historical 30-day repeat-purchase rate: 28.03%


### Historical Repeat-Purchase Baseline — Interpretation

At the 1 June 2011 historical cutoff:

- **2,658 customers** were eligible based on activity during the preceding 180 days;
- **745 customers** made another qualifying merchandise purchase within the following 30 days;
- the historical 30-day repeat-purchase rate was **28.03%**.

This rate is used as the expected control-group baseline for experiment power planning.

It is only a planning estimate derived from historical observational data. It is not a randomized control-group result and does not measure the effect of any retention intervention.

## 6.9 Power Analysis and Required Sample Size

Power analysis estimates how many customers would be required to detect a meaningful improvement in 30-day repeat purchase.

The historical control-rate assumption is:

**28.03%**

The proposed experiment uses:

- **two-sided significance level:** 5%;
- **statistical power:** 80%;
- **50/50 control-treatment allocation**.

Because the project does not have a real business stakeholder defining a minimum worthwhile uplift, several absolute minimum detectable effects are evaluated rather than presenting one arbitrary value as a business requirement.

Sample-size requirements are calculated for treatment improvements of:

- +1 percentage point;
- +2 percentage points;
- +3 percentage points;
- +5 percentage points.

In [11]:
from statsmodels.stats.power import (
    NormalIndPower,
)
from statsmodels.stats.proportion import (
    proportion_effectsize,
)


power_analysis = NormalIndPower()

control_rate = baseline_repeat_rate

mde_percentage_points = [
    1,
    2,
    3,
    5,
]


power_rows = []


for mde_pp in mde_percentage_points:
    treatment_rate = (
        control_rate
        + mde_pp / 100
    )

    effect_size = abs(
        proportion_effectsize(
            treatment_rate,
            control_rate,
        )
    )

    required_per_group = (
        power_analysis.solve_power(
            effect_size=effect_size,
            power=0.80,
            alpha=0.05,
            ratio=1.0,
            alternative="two-sided",
        )
    )

    required_per_group = int(
        np.ceil(required_per_group)
    )

    power_rows.append(
        {
            "absolute_uplift_pp": mde_pp,
            "control_rate_pct": (
                control_rate * 100
            ),
            "treatment_rate_pct": (
                treatment_rate * 100
            ),
            "required_per_group": (
                required_per_group
            ),
            "required_total": (
                required_per_group * 2
            ),
        }
    )


power_table = pd.DataFrame(
    power_rows
)


display(
    power_table.round(2)
)

,absolute_uplift_pp,control_rate_pct,treatment_rate_pct,required_per_group,required_total
0,1,28.03,29.03,32006,64012
1,2,28.03,30.03,8083,16166
2,3,28.03,31.03,3628,7256
3,5,28.03,33.03,1330,2660


### Power Analysis — Interpretation

The historical 30-day repeat-purchase rate of **28.03%** is used as the expected control-group baseline.

At 80% power, a two-sided 5% significance level, and equal treatment allocation, the required sample size depends strongly on the minimum detectable effect.

For example:

- a **1 percentage-point** uplift requires approximately **64,012 customers**;
- a **2 percentage-point** uplift requires approximately **16,166 customers**;
- a **3 percentage-point** uplift requires approximately **7,256 customers**;
- a **5 percentage-point** uplift requires approximately **2,660 customers**.

For planning purposes, a **3 percentage-point absolute uplift** is selected as the minimum detectable effect.

This corresponds to increasing the 30-day repeat-purchase rate from approximately **28.03% to 31.03%**, or about a **10.7% relative improvement**.

The experiment would therefore require approximately:

- **3,628 customers per group**
- **7,256 customers in total**

The MDE represents a planning assumption about the smallest commercially meaningful effect and is not an observed campaign result.

### Feasibility at the Historical Eligible Population Size

The historical cutoff contained 2,658 eligible customers, which is below the sample size required to detect a 3 percentage-point uplift with 80% power.

The next calculation estimates the approximate minimum effect that an equally split experiment of this size could detect.

In [12]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from scipy.optimize import brentq


available_total = len(
    baseline_customers
)

available_per_group = (
    available_total // 2
)

power_analysis = NormalIndPower()


def achieved_power(
    treatment_rate
):
    effect_size = abs(
        proportion_effectsize(
            treatment_rate,
            baseline_repeat_rate,
        )
    )

    return power_analysis.power(
        effect_size=effect_size,
        nobs1=available_per_group,
        alpha=0.05,
        ratio=1.0,
        alternative="two-sided",
    )


detectable_treatment_rate = brentq(
    lambda rate: (
        achieved_power(rate) - 0.80
    ),
    baseline_repeat_rate,
    min(
        baseline_repeat_rate + 0.20,
        0.999,
    ),
)


detectable_uplift_pp = (
    detectable_treatment_rate
    - baseline_repeat_rate
) * 100


relative_uplift_pct = (
    detectable_treatment_rate
    / baseline_repeat_rate
    - 1
) * 100


print(
    "Available total customers:",
    available_total,
)

print(
    "Approx. customers per group:",
    available_per_group,
)

print(
    "Detectable treatment rate:",
    f"{detectable_treatment_rate * 100:.2f}%",
)

print(
    "Approx. detectable absolute uplift:",
    f"{detectable_uplift_pp:.2f} percentage points",
)

print(
    "Approx. detectable relative uplift:",
    f"{relative_uplift_pct:.2f}%",
)

Available total customers: 2658
Approx. customers per group: 1329
Detectable treatment rate: 33.03%
Approx. detectable absolute uplift: 5.00 percentage points
Approx. detectable relative uplift: 17.84%


### Experiment Feasibility — Interpretation

At the historical cutoff, the eligible customer population contains **2,658 customers**, or approximately **1,329 customers per experimental group** under equal allocation.

With this sample size, an experiment would have approximately 80% power to detect an increase in 30-day repeat purchase from:

**28.03% to approximately 33.03%**

This corresponds to:

- **5.00 percentage points absolute uplift**
- approximately **17.84% relative uplift**

This is substantially larger than the **3 percentage-point minimum detectable effect** selected for planning.

Therefore, a campaign launched on a population similar in size to this historical snapshot would be underpowered for detecting the smaller effect considered commercially meaningful.

A stronger experimental design would accumulate approximately **7,256 eligible customers** before evaluating the 3 percentage-point target effect, rather than lowering the statistical standard after the experiment begins.

## 6.9 Retailer Experiment Recommendation

The proposed retention-offer experiment should use:

- **Unit of randomization:** customer
- **Allocation:** 50% control / 50% treatment
- **Control:** normal customer experience with no retention offer
- **Treatment:** receive the retention offer
- **Primary metric:** repeat purchase within 30 days
- **Significance level:** 5%, two-sided
- **Target power:** 80%
- **Historical control-rate assumption:** 28.03%
- **Minimum detectable effect:** +3 percentage points
- **Target treatment rate:** 31.03%
- **Required sample:** approximately 3,628 customers per group
- **Required total sample:** approximately 7,256 customers

Guardrail metrics should include merchandise revenue per randomized customer, average order value, order frequency, cancellation or return behavior where available, campaign cost, and net incremental value.

The historical June 2011 cutoff contains only 2,658 eligible customers. At that population size, the experiment would be powered only for an effect of approximately **+5 percentage points**.

The recommended approach is therefore to **accumulate a larger eligible population or combine multiple future enrollment cohorts while preserving the same randomized assignment and measurement protocol**.

No actual retailer campaign was run in this project. These calculations represent an experiment design and power-planning exercise only.

## 6.10 Phase 6 Summary

Phase 6 demonstrates both retrospective A/B-test analysis and prospective experiment design.

### Public Randomized Experiment

The Cookie Cats experiment compared `gate_30` with `gate_40`.

The primary 7-day retention outcome showed:

- **gate_30:** 19.02%
- **gate_40:** 18.20%
- **Absolute effect:** -0.82 percentage points
- **Relative effect:** -4.31%
- **95% CI:** -1.33 to -0.31 percentage points
- **p-value:** 0.0016
- **Cohen's h:** -0.0211

The evidence therefore supports a **do-not-ship recommendation for `gate_40`** when the objective is improving 7-day retention.

The secondary 1-day retention outcome also moved negatively, but the difference was not statistically conclusive.

### Proposed Retail Retention Experiment

For the retailer, a hypothetical retention-offer experiment was designed with:

- customer-level random assignment;
- a 50/50 control-treatment split;
- **30-day repeat purchase** as the primary outcome;
- business and operational guardrail metrics;
- 80% statistical power;
- a two-sided 5% significance level.

Historical retail behavior produced an estimated control repeat-purchase rate of **28.03%**.

Using a minimum detectable effect of **+3 percentage points**, the proposed experiment would require approximately:

**7,256 customers in total**

The historical cutoff contained only 2,658 eligible customers, which would provide sufficient power only for a larger effect of approximately **+5 percentage points**.

The retailer experiment is a planning exercise only. No real treatment was applied to the retail dataset, so no causal claim is made about retention offers or retailer customer behavior.

Causal interpretation in this phase is limited to the public randomized Cookie Cats experiment.